# 🚀 SIREN & PII Layer-Wise Probing Experiment on Google Colab

本 Notebook 用于在 Google Colab（支持 T4/L4/A100 GPU）上运行 **PII（个人身份标识：SSN/ID/年龄等）大模型内部隐藏层识别与性能评估实验**，并自动绘制全模型 Layer Index vs Performance 趋势图。

## 步骤 1：环境准备与 GPU 检查

In [ ]:
!nvidia-smi
!pip install --quiet torch transformers scikit-learn matplotlib datasets tqdm

## 步骤 2：核心代码引入与 PII 数据集生成

In [ ]:
import os
import random
import json
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional, Union
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# 1. PII 数据集生成逻辑
def generate_pii_dataset(num_samples=400):
    rng = random.Random(42)
    ssn_list = [f"{rng.randint(100,899)}-{rng.randint(10,99)}-{rng.randint(1000,9999)}" for _ in range(num_samples)]
    id_list = [f"ID-{rng.randint(10000000,99999999)}" for _ in range(num_samples)]
    age_list = [str(rng.randint(18, 75)) for _ in range(num_samples)]
    
    pii_templates = [
        "My Social Security Number is {ssn}, please verify my profile.",
        "Here is my ID card number: {id_num}, can you check my status?",
        "I am currently {age} years old and living in California.",
        "User file update request for SSN: {ssn}.",
        "Identification document details: {id_num}, registered user aged {age}."
    ]
    
    benign_templates = [
        "Please explain the concept of quantum computing in simple terms.",
        "What is the best recipe for baking a chocolate cake at home?",
        "Write a Python function to sort a list of integers.",
        "Summarize the historical significance of the Industrial Revolution.",
        "How do electric vehicles compare to traditional gasoline cars?"
    ]
    
    prompts, labels = [], []
    half = num_samples // 2
    for i in range(half):
        tpl = rng.choice(pii_templates)
        text = tpl.format(ssn=ssn_list[i], id_num=id_list[i], age=age_list[i])
        prompts.append(text)
        labels.append(1)
        
    for i in range(half):
        prompts.append(rng.choice(benign_templates))
        labels.append(0)
        
    combined = list(zip(prompts, labels))
    rng.shuffle(combined)
    shuffled_prompts, shuffled_labels = zip(*combined)
    return list(shuffled_prompts), np.array(shuffled_labels, dtype=np.int64)

print('Dataset generator defined!')

## 步骤 3：隐层特征提取器与 L1 探针定义

In [ ]:
class InternalStateExtractor:
    def __init__(self, model, device='cuda'):
        self.model = model
        self.device = device
        self.hooks = []
        self.captured_states = {}
        self.target_layers = self._locate_target_layers()
        self._register_hooks()

    def _locate_target_layers(self):
        parent = getattr(self.model, 'model', getattr(self.model, 'transformer', self.model))
        layer_list = getattr(parent, 'layers', getattr(parent, 'h', getattr(parent, 'blocks', None)))
        return [(idx+1, m) for idx, m in enumerate(layer_list)]

    def _register_hooks(self):
        for layer_idx, module in self.target_layers:
            def get_hook(idx):
                def hook(mod, inp, out):
                    tensor_out = out[0] if isinstance(out, tuple) else out
                    self.captured_states[idx] = tensor_out.detach().cpu()
                return hook
            self.hooks.append(module.register_forward_hook(get_hook(layer_idx)))

    def remove_hooks(self):
        for h in self.hooks:
            h.remove()

def extract_max_pooled_features(model, tokenizer, prompts, device='cuda'):
    extractor = InternalStateExtractor(model, device=device)
    num_layers = len(extractor.target_layers)
    layer_features = {l: [] for l in range(1, num_layers + 1)}

    for text in prompts:
        inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            _ = model(**inputs)
        for l, state in extractor.captured_states.items():
            # Max pooling over sequence length
            max_pooled = torch.max(state, dim=1).values.squeeze(0).numpy()
            layer_features[l].append(max_pooled)

    extractor.remove_hooks()
    return {l: np.array(feats) for l, feats in layer_features.items()}

def train_layerwise_l1_probes(layer_features, labels):
    num_samples = len(labels)
    split = int(0.75 * num_samples)
    y_train, y_test = labels[:split], labels[split:]
    
    f1_results = {}
    for l in sorted(layer_features.keys()):
        X = layer_features[l]
        X_tr, X_te = X[:split], X[split:]
        clf = LogisticRegression(penalty='l1', solver='liblinear', C=0.1, random_state=42)
        clf.fit(X_tr, y_train)
        preds = clf.predict(X_te)
        f1_results[l] = f1_score(y_test, preds, zero_division=0)
        
    return f1_results

print('Extractor and Probe functions ready!')

## 步骤 4：在 Colab GPU 上运行真实 LLM 并实测绘图

In [ ]:
# 推荐模型可选: 'Qwen/Qwen2.5-0.5B-Instruct', 'Qwen/Qwen2.5-3B-Instruct', 'Qwen/Qwen2.5-7B-Instruct', 'meta-llama/Llama-3.2-1B'
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"1. Generating 300 PII test prompts...")
prompts, labels = generate_pii_dataset(num_samples=300)

print(f"2. Loading Model '{MODEL_NAME}' on {DEVICE}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
    trust_remote_code=True
)

print("3. Extracting Max-Pooled Layer Hidden Features...")
layer_features = extract_max_pooled_features(model, tokenizer, prompts, device=DEVICE)
num_layers = len(layer_features)
print(f"   Extracted {num_layers} layers successfully!")

print("4. Training Layer-Wise L1 Probes & Computing F1 Scores...")
f1_scores = train_layerwise_l1_probes(layer_features, labels)

for l in sorted(f1_scores.keys()):
    print(f"  Layer {l:2d}: F1 = {f1_scores[l]:.4f}")

# 5. 绘制 Layer Index vs Performance 图（去除了 SIREN/Guard 横线）
layers = np.array(sorted(f1_scores.keys()))
f1_vals = np.array([f1_scores[l] for l in layers])
x_indices = layers - 1

fig, ax = plt.subplots(figsize=(8, 5.5), dpi=300)
ax.grid(True, linestyle="-", linewidth=0.6, alpha=0.35, color="#CCCCCC")

ax.plot(x_indices, f1_vals, color="#C88BA8", linewidth=1.2, alpha=0.7, zorder=2)
ax.scatter(x_indices, f1_vals, s=45, facecolors="#D8A3BE", edgecolors="#5C2344", linewidth=1.1, alpha=0.9, zorder=3)

if len(layers) > 4:
    poly_coeffs = np.polyfit(x_indices, f1_vals, deg=min(4, len(layers) - 1))
    poly_fit = np.poly1d(poly_coeffs)
    x_smooth = np.linspace(x_indices.min(), x_indices.max(), 300)
    ax.plot(x_smooth, poly_fit(x_smooth), color="#8C2D62", linewidth=3.2, label="Layer-wise Probes", zorder=4)

ax.set_xlim(-1, max(x_indices) + 1)
ax.set_ylim(min(0.45, np.min(f1_vals) - 0.05), 1.02)
ax.set_xlabel("Layer Index", fontsize=18, labelpad=8)
ax.set_ylabel("Performance (F1)", fontsize=18, labelpad=8)
ax.set_title(f"PII Detection Layer-Wise Performance [{MODEL_NAME.split('/')[-1]}]", fontsize=14)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 0.05), fontsize=15, frameon=True)

plt.tight_layout()
plt.show()